# Temporary M2 CelebA download, sample, and validation

This notebook keeps the official CelebA ZIP in a **private Google Drive folder**. It processes data in temporary Colab storage and copies only small validation artifacts back to Drive.

Before running it, read and accept the official CelebA agreement. CelebA is for non-commercial research use and must not be redistributed. Do not share the Drive data folder, review archive, raw images, or generated split filenames publicly.

The attributes, sample size, seed, split fractions, and corruption values below are placeholders. This notebook does not freeze the experiment protocol. A CPU runtime is enough.

In [16]:
from pathlib import Path

# You must change this to True after reading and accepting the official terms.
ACCEPT_CELEBA_TERMS = True

REPOSITORY_URL = "https://github.com/Omkar38/mixed-precision-visual-shift.git"
BRANCH = "Gauri-Commits"  # Change if the sampler is published on another branch.
RUN_NAME = "celeba-placeholder-v1"

ATTRIBUTES = ["Eyeglasses", "Wearing_Hat", "Smiling"]
PER_LABEL = 200
SAMPLE_SEED = 590
SPLIT_SEED = 590
PILOT_FRACTION = 1 / 12
DEVELOPMENT_FRACTION = 1 / 2
TEST_FRACTION = 5 / 12

# Target folder supplied by Kaushal. Colab needs a My Drive shortcut path, not this URL.
TARGET_DRIVE_FOLDER_ID = "1ZV28vSNXy4Y5UkiCq7OHt3u1DaPdWK2u"
TARGET_DRIVE_FOLDER_URL = (
    f"https://drive.google.com/drive/folders/{TARGET_DRIVE_FOLDER_ID}"
)
DRIVE_FOLDER_NAME = "mixed-precision-visual-shift"
DRIVE_FOLDER_CANDIDATES = [
    Path("/content/drive/Shareddrives") / DRIVE_FOLDER_NAME,  # Workspace Shared Drive
    Path("/content/drive/MyDrive") / DRIVE_FOLDER_NAME,      # My Drive shortcut
    Path("/content/drive/.shortcut-targets-by-id")
    / TARGET_DRIVE_FOLDER_ID
    / DRIVE_FOLDER_NAME,                                       # Resolved shortcut target
]
DRIVE_PRIVATE_ROOT = None
DRIVE_SOURCE_DIR = None
DRIVE_EXPORT_DIR = None
LOCAL_SOURCE_DIR = Path("/content/celeba-source")
LOCAL_SELECTED_ROOT = Path("/content/celeba-selected")
LOCAL_RUN_DIR = Path("/content/mpvs-m2") / RUN_NAME
REPO_DIR = Path("/content/mixed-precision-visual-shift")

print("Placeholder records requested:", len(ATTRIBUTES) * 2 * PER_LABEL)
print("Target Drive folder:", TARGET_DRIVE_FOLDER_URL)
print("Expected folder name:", DRIVE_FOLDER_NAME)
print("The mounted path will be detected after Drive is mounted.")

Placeholder records requested: 1200
Target Drive folder: https://drive.google.com/drive/folders/1ZV28vSNXy4Y5UkiCq7OHt3u1DaPdWK2u
Expected folder name: mixed-precision-visual-shift
The mounted path will be detected after Drive is mounted.


## 1. Mount private Google Drive

Only run this notebook from an account and notebook you trust. Mounted Drive contents are available to notebook code.

The target is named `mixed-precision-visual-shift`. The notebook checks the Workspace Shared Drive path, the visible My Drive shortcut path, and DriveFS's hidden `.shortcut-targets-by-id/<folder-id>/mixed-precision-visual-shift` path.

The **Shared with me** page is not mounted as a `shared-with-me` directory. If the target is a shared folder rather than a Workspace Shared Drive, open it, choose **Organize → Add shortcut**, place the shortcut directly under **My Drive**, and keep the name `mixed-precision-visual-shift`.

In [22]:
# if not ACCEPT_CELEBA_TERMS:
#     raise RuntimeError(
#         "Read the official CelebA agreement, then set ACCEPT_CELEBA_TERMS=True."
#     )

# from google.colab import drive

# drive.mount("/content/drive")
# matches = [path for path in DRIVE_FOLDER_CANDIDATES if path.is_dir()]
# if not matches:
#     raise FileNotFoundError(
#         f"Could not find {DRIVE_FOLDER_NAME!r} in a mounted Shared Drive or My Drive. "
#         f"Open {TARGET_DRIVE_FOLDER_URL}. If it appears under 'Shared with me', "
#         "add a shortcut directly under My Drive and rerun this cell."
#     )
# DRIVE_PRIVATE_ROOT = matches[0]
# DRIVE_SOURCE_DIR = DRIVE_PRIVATE_ROOT / "celeba"
# DRIVE_EXPORT_DIR = DRIVE_PRIVATE_ROOT / "artifacts" / RUN_NAME

# def create_directory_if_missing(path):
#     if path.is_dir():
#         print("Using existing directory:", path)
#         return
#     if path.exists():
#         raise FileExistsError(f"Expected a directory, but found a file: {path}")
#     path.mkdir(parents=True)
#     print("Created directory:", path)

# for directory in (
#     DRIVE_SOURCE_DIR,
#     LOCAL_SOURCE_DIR,
#     LOCAL_SELECTED_ROOT,
#     LOCAL_RUN_DIR,
# ):
#     create_directory_if_missing(directory)

# print("Using target folder:", DRIVE_PRIVATE_ROOT)
# print("Source folder:", DRIVE_SOURCE_DIR)
# print("Export folder:", DRIVE_EXPORT_DIR)

## 2. Download and verify the official source files

The file IDs and expected MD5 values match the current `torchvision.datasets.CelebA` source. The notebook tries the official Google Drive files first. If Google applies its public quota, it uses the clearly labelled Freie Universität Berlin academic mirror and accepts a file only when it has the same published MD5. The CelebA agreement still applies.

In [23]:
import hashlib
import shutil
import subprocess
import sys

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "gdown"], check=True)
import gdown

OFFICIAL_CELEBA_PAGE = "https://mmlab.ie.cuhk.edu.hk/projects/CelebA.html"
ACADEMIC_MIRROR_BASE_URL = "https://ftp.mi.fu-berlin.de/pub/cmb-data/celeba"

# Download the large, quota-prone image archive last so the two small
# metadata files can still be downloaded and verified first.
FILES = {
    "list_attr_celeba.txt": {
        "id": "0B7EVK8r0v71pblRyaVFSWGxPY0U",
        "md5": "75e246fa4810816ffd6ee81facbd244c",
    },
    "identity_CelebA.txt": {
        "id": "1_ee_0u7vcNLOfNLegJRHmolfH5ICW-XS",
        "md5": "32bd1bd63d3c78cd57e08160ec5ed1e2",
    },
    "img_align_celeba.zip": {
        "id": "0B7EVK8r0v71pZjFTYXZWM3FlRnM",
        "md5": "00d2c5bc6d35e252742224ab0c1e8fcb",
    },
}

def digest(path, algorithm):
    value = hashlib.new(algorithm)
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            value.update(chunk)
    return value.hexdigest()

def download_from_checksum_verified_mirror(filename, destination, expected_md5):
    mirror_url = f"{ACADEMIC_MIRROR_BASE_URL}/{filename}"
    temporary = Path("/content") / f"{filename}.part"
    print("Official Google Drive download unavailable.")
    print("Trying checksum-verified academic mirror:", mirror_url)
    subprocess.run(
        [
            "wget", "--continue", "--show-progress",
            "--output-document", str(temporary), mirror_url,
        ],
        check=True,
    )
    mirror_md5 = digest(temporary, "md5")
    if mirror_md5 != expected_md5:
        raise RuntimeError(
            f"Mirror MD5 mismatch for {filename}: expected {expected_md5}, "
            f"got {mirror_md5}. The file was not copied to Drive."
        )
    shutil.copy2(temporary, destination)
    temporary.unlink()
    print("Verified and saved to Drive:", destination)
    return str(destination)

source_hashes = {}
for filename, details in FILES.items():
    destination = DRIVE_SOURCE_DIR / filename
    if not destination.exists():
        result = None
        try:
            result = gdown.download(
                id=details["id"], output=str(destination), quiet=False
            )
        except Exception as error:
            print(f"Official download failed for {filename}: {type(error).__name__}")
        if result is None:
            result = download_from_checksum_verified_mirror(
                filename, destination, details["md5"]
            )
    actual_md5 = digest(destination, "md5")
    if actual_md5 != details["md5"]:
        raise RuntimeError(
            f"MD5 mismatch for {destination}: expected {details['md5']}, got {actual_md5}. "
            "Do not use or silently overwrite this file."
        )
    source_hashes[filename] = {
        "md5": actual_md5,
        "sha256": digest(destination, "sha256"),
        "bytes": destination.stat().st_size,
    }
    print(filename, source_hashes[filename])

print("All three source files passed their expected MD5 checks.")

RuntimeError: Could not download list_attr_celeba.txt. Open https://mmlab.ie.cuhk.edu.hk/projects/CelebA.html, place the official file at /content/drive/.shortcut-targets-by-id/1ZV28vSNXy4Y5UkiCq7OHt3u1DaPdWK2u/mixed-precision-visual-shift/celeba/list_attr_celeba.txt, and rerun this cell.

## 3. Copy the archive to temporary Colab storage and inspect it

We do not extract 202,599 files into Drive. The notebook copies one ZIP to `/content`, checks its entries, and later extracts only the selected images.

In [ ]:
import shutil
import zipfile

for filename in FILES:
    shutil.copy2(DRIVE_SOURCE_DIR / filename, LOCAL_SOURCE_DIR / filename)

archive_path = LOCAL_SOURCE_DIR / "img_align_celeba.zip"
with zipfile.ZipFile(archive_path) as archive:
    broken_member = archive.testzip()
    if broken_member is not None:
        raise RuntimeError(f"Broken ZIP member: {broken_member}")
    jpeg_members = [
        name for name in archive.namelist()
        if name.startswith("img_align_celeba/") and name.lower().endswith(".jpg")
    ]

if len(jpeg_members) != 202599:
    raise RuntimeError(f"Expected 202599 JPEG entries, found {len(jpeg_members)}")
if len({Path(name).name for name in jpeg_members}) != len(jpeg_members):
    raise RuntimeError("The archive contains duplicate JPEG basenames.")

print("ZIP integrity passed. JPEG entries:", len(jpeg_members))
print("No full extraction was performed.")

## 4. Clone the project branch and install it

The branch must contain `src/mpvs/data/sampling.py` and the `m2-verify-images` command. Commit and push the local sampler changes before running this notebook.

In [ ]:
def run(command, *, cwd=None):
    print("+", " ".join(map(str, command)))
    return subprocess.run(command, cwd=cwd, check=True, text=True)

if not REPO_DIR.exists():
    run(["git", "clone", "--branch", BRANCH, "--single-branch", REPOSITORY_URL, str(REPO_DIR)])
else:
    run(["git", "fetch", "origin", BRANCH], cwd=REPO_DIR)
    run(["git", "switch", BRANCH], cwd=REPO_DIR)
    run(["git", "pull", "--ff-only", "origin", BRANCH], cwd=REPO_DIR)

required = [REPO_DIR / "pyproject.toml", REPO_DIR / "src/mpvs/data/sampling.py"]
missing = [str(path.relative_to(REPO_DIR)) for path in required if not path.exists()]
if missing:
    raise RuntimeError(
        f"Branch {BRANCH!r} does not contain the current M2 sampler: missing {missing}. "
        "Commit and push the local changes, then start a fresh runtime."
    )

run([sys.executable, "-m", "pip", "install", "-e", str(REPO_DIR)])
commit = subprocess.run(
    ["git", "rev-parse", "HEAD"], cwd=REPO_DIR, check=True, text=True, capture_output=True
).stdout.strip()
help_result = subprocess.run(
    [sys.executable, "-m", "mpvs.cli", "m2-verify-images", "--help"],
    check=True, text=True, capture_output=True,
)
print("Repository commit:", commit)
print(help_result.stdout.splitlines()[0])

## 5. Normalize annotations without extracting images

This creates a temporary annotation-only manifest. It is not a final manifest because it does not yet contain image checksums.

In [ ]:
normalized_unverified = LOCAL_RUN_DIR / "celeba-normalized-unverified.jsonl"
normalize_command = [
    sys.executable, "-m", "mpvs.cli", "m2-normalize-celeba",
    "--attributes", str(LOCAL_SOURCE_DIR / "list_attr_celeba.txt"),
    "--identities", str(LOCAL_SOURCE_DIR / "identity_CelebA.txt"),
    "--image-root", str(LOCAL_SELECTED_ROOT),
    "--skip-image-checksums",
]
questions = {
    "Eyeglasses": "Is this person wearing eyeglasses?",
    "Wearing_Hat": "Is this person wearing a hat?",
    "Smiling": "Is this person smiling?",
}
for attribute in ATTRIBUTES:
    normalize_command.extend(["--attribute-question", f"{attribute}={questions[attribute]}"])
normalize_command.extend(["--output", str(normalized_unverified)])
run(normalize_command, cwd=REPO_DIR)
print("Temporary normalized bytes:", normalized_unverified.stat().st_size)

## 6. Create the placeholder balanced sample

This first pass is allowed to lack checksums because the next cells extract and verify exactly the selected images.

In [ ]:
sample_unverified = LOCAL_RUN_DIR / "celeba-sampled-unverified.jsonl"
sample_command = [
    sys.executable, "-m", "mpvs.cli", "m2-sample-celeba",
    "--input", str(normalized_unverified),
    "--per-label", str(PER_LABEL),
    "--seed", str(SAMPLE_SEED),
    "--allow-missing-image-checksums",
]
for attribute in ATTRIBUTES:
    sample_command.extend(["--attribute", attribute])
sample_command.extend(["--output", str(sample_unverified)])
run(sample_command, cwd=REPO_DIR)

import json
sample_rows = [json.loads(line) for line in sample_unverified.read_text().splitlines() if line]
expected_records = len(ATTRIBUTES) * 2 * PER_LABEL
if len(sample_rows) != expected_records:
    raise RuntimeError(f"Expected {expected_records} sampled rows, found {len(sample_rows)}")
print("Balanced annotation-only sample rows:", len(sample_rows))

## 7. Extract only selected images and attach checksums

This is the storage-saving step. It extracts about 1,200 selected JPEGs instead of all 202,599 images.

In [ ]:
selected_filenames = sorted({row["image_relpath"] for row in sample_rows})
if len(selected_filenames) != len(sample_rows):
    raise RuntimeError("Sampler output reused a source image.")

member_by_filename = {Path(name).name: name for name in jpeg_members}
missing_from_zip = sorted(set(selected_filenames) - set(member_by_filename))
if missing_from_zip:
    raise RuntimeError(f"Selected images missing from ZIP: {missing_from_zip[:10]}")

with zipfile.ZipFile(archive_path) as archive:
    for filename in selected_filenames:
        target = LOCAL_SELECTED_ROOT / filename
        with archive.open(member_by_filename[filename]) as source, target.open("wb") as output:
            shutil.copyfileobj(source, output)

extracted = list(LOCAL_SELECTED_ROOT.glob("*.jpg"))
if len(extracted) != len(selected_filenames):
    raise RuntimeError(f"Expected {len(selected_filenames)} extracted images, found {len(extracted)}")
print("Selected images extracted:", len(extracted))

sample_verified = LOCAL_RUN_DIR / "celeba-sampled-verified.jsonl"
run([
    sys.executable, "-m", "mpvs.cli", "m2-verify-images",
    "--input", str(sample_unverified),
    "--image-root", str(LOCAL_SELECTED_ROOT),
    "--output", str(sample_verified),
], cwd=REPO_DIR)
verified_rows = [json.loads(line) for line in sample_verified.read_text().splitlines() if line]
if any(not row.get("source_image_sha256") for row in verified_rows):
    raise RuntimeError("At least one selected image is missing its SHA-256 checksum.")
print("Verified sampled images:", len(verified_rows))

## 8. Create and validate placeholder identity-disjoint splits

The requested fractions are placeholders. Whole identities take priority, so achieved question counts may differ.

In [ ]:
split_dir = LOCAL_RUN_DIR / "splits"
run([
    sys.executable, "-m", "mpvs.cli", "m2-split",
    "--input", str(sample_verified),
    "--output-dir", str(split_dir),
    "--seed", str(SPLIT_SEED),
    "--pilot-fraction", str(PILOT_FRACTION),
    "--development-fraction", str(DEVELOPMENT_FRACTION),
    "--test-fraction", str(TEST_FRACTION),
], cwd=REPO_DIR)
run([
    sys.executable, "-m", "mpvs.cli", "m2-validate-splits",
    str(split_dir / "pilot.jsonl"),
    str(split_dir / "development.jsonl"),
    str(split_dir / "test.jsonl"),
], cwd=REPO_DIR)
split_summary = json.loads((split_dir / "split-summary.json").read_text())
print(json.dumps(split_summary["validation"]["splits"], indent=2))

## 9. Build a placeholder pilot review package

These corruption values are only candidates. The generated review page uses no model answers. A person must review it before any severity is accepted.

In [ ]:
corruption_specs = [
    {"kind": "clean", "severity_id": "clean", "value": None},
    {"kind": "darkness", "severity_id": "dark-1-placeholder", "value": 0.60},
    {"kind": "darkness", "severity_id": "dark-2-placeholder", "value": 0.35},
    {"kind": "darkness", "severity_id": "dark-3-placeholder", "value": 0.15},
    {"kind": "gaussian_blur", "severity_id": "blur-1-placeholder", "value": 0.5},
    {"kind": "gaussian_blur", "severity_id": "blur-2-placeholder", "value": 1.0},
    {"kind": "gaussian_blur", "severity_id": "blur-3-placeholder", "value": 2.0},
]
corruption_path = LOCAL_RUN_DIR / "placeholder-corruptions.json"
corruption_path.write_text(json.dumps(corruption_specs, indent=2) + "\n")
pilot_variants = LOCAL_RUN_DIR / "pilot-variants.jsonl"
run([
    sys.executable, "-m", "mpvs.cli", "m2-expand-variants",
    "--input", str(split_dir / "pilot.jsonl"),
    "--corruptions", str(corruption_path),
    "--output", str(pilot_variants),
], cwd=REPO_DIR)
review_dir = LOCAL_RUN_DIR / "pilot-review"
run([
    sys.executable, str(REPO_DIR / "scripts/m2_pilot_review.py"),
    "--variants", str(pilot_variants),
    "--image-root", str(LOCAL_SELECTED_ROOT),
    "--output-dir", str(review_dir),
], cwd=REPO_DIR)
review_zip = shutil.make_archive(str(LOCAL_RUN_DIR / "pilot-review-private"), "zip", review_dir)
print("Private review package:", review_zip)
print("Unzip it locally and open index.html. Do not share it publicly.")

## 10. Export only small private artifacts to Drive

The full annotation-only manifest and extracted images remain temporary. The export contains the verified sample, split manifests, metadata, placeholder pilot variants, and a private zipped review package.

In [ ]:
from datetime import datetime, timezone

if DRIVE_EXPORT_DIR.exists():
    raise FileExistsError(
        f"{DRIVE_EXPORT_DIR} already exists. Change RUN_NAME to preserve the earlier run."
    )
DRIVE_EXPORT_DIR.mkdir(parents=True)

files_to_copy = [
    normalized_unverified.with_suffix(normalized_unverified.suffix + ".metadata.json"),
    sample_unverified.with_suffix(sample_unverified.suffix + ".metadata.json"),
    sample_verified,
    sample_verified.with_suffix(sample_verified.suffix + ".metadata.json"),
    corruption_path,
    pilot_variants,
    pilot_variants.with_suffix(pilot_variants.suffix + ".metadata.json"),
    Path(review_zip),
]
for source in files_to_copy:
    shutil.copy2(source, DRIVE_EXPORT_DIR / source.name)
shutil.copytree(split_dir, DRIVE_EXPORT_DIR / "splits")

run_summary = {
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "repository_url": REPOSITORY_URL,
    "branch": BRANCH,
    "commit": commit,
    "status": "placeholder_not_frozen",
    "source_files": source_hashes,
    "zip_jpeg_entry_count": len(jpeg_members),
    "attributes": ATTRIBUTES,
    "per_label": PER_LABEL,
    "sample_seed": SAMPLE_SEED,
    "split_seed": SPLIT_SEED,
    "split_fractions": {
        "pilot": PILOT_FRACTION,
        "development": DEVELOPMENT_FRACTION,
        "test": TEST_FRACTION,
    },
    "corruption_specs": corruption_specs,
    "selected_image_count": len(selected_filenames),
    "human_answerability_review_complete": False,
}
(DRIVE_EXPORT_DIR / "run-summary.json").write_text(
    json.dumps(run_summary, indent=2, sort_keys=True) + "\n"
)

print("Export complete:", DRIVE_EXPORT_DIR)
print("Files exported:")
for path in sorted(DRIVE_EXPORT_DIR.rglob("*")):
    if path.is_file():
        print(" -", path.relative_to(DRIVE_EXPORT_DIR), path.stat().st_size, "bytes")

## What remains after the notebook

1. Download the private `pilot-review-private.zip`, unzip it, and open `index.html`.
2. Review labels and corruption answerability without model outputs.
3. Save the decisions JSONL privately and validate it with `m2-validate-answerability`.
4. The team must approve the attributes, counts, seeds, split sizes, and severities.
5. Rerun into a new versioned `RUN_NAME` after approval.
6. Do not call this placeholder run a final manifest or an experimental result.